In [ ]:
import os
DATA_ROOT = os.environ.get('WRH_DATA_ROOT', 'G:/WRH_data')  # local data folder; see README
from pathlib import Path
import json, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.stats import norm
import statsmodels.api as sm
import statsmodels.formula.api as smf
warnings.filterwarnings('ignore')
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 80)

ACC_FP  = Path(rf'{DATA_ROOT}/AIS_processing/accident_data/accident_all_weather_2011_2021_age.parquet')
OUT_DIR = Path(os.environ.get('M3_OUT_DIR', 'M3_reanalysis_outputs'))  # folder written by M3_reanalysis_complete.py
OUT_DIR.mkdir(parents=True, exist_ok=True)

FE_MODE = 'area_month'            # 'area_month' | 'g10' | 'g10_month'
USE_CARGO_SUBTYPE_CONTROL = True
RUN_FULL_INTERACTION = True
RUN_NONLINEAR = False
RUN_GLM_SENSITIVITY = False

CLUSTER_COL = 'g5'
OUTCOMES = {'ser': 'Serious outcome', 'tloss': 'Total loss'}
TOTAL_LOSS_CATEGORIES = {'actual total loss', 'constructive total loss'}
WAVE_EDGES = [0.0, 1.25, 2.5, 4.0, 6.0, np.inf]
WAVE_LABELS = ['<1.25', '1.25-2.5', '2.5-4', '4-6', '>=6']
WIND_EDGES = [0.0, 8.0, 10.8, 13.9, 17.2, np.inf]
WIND_LABELS = ['<8', '8-10.8', '10.8-13.9', '13.9-17.2', '>=17.2']
HAZARDS = {
    'wv': dict(name='Significant wave height', symbol=r'$H_s$ (m)', reference=WAVE_LABELS[0],
               labels=WAVE_LABELS, colour_light='#8fb9d1', colour_dark='#1a4a63'),
    'wd': dict(name='10-m wind speed', symbol=r'$U_{10}$ (m s$^{-1}$)', reference=WIND_LABELS[0],
               labels=WIND_LABELS, colour_light='#e9a58c', colour_dark='#93402a'),
}
MARKERS = {'ser': 's', 'tloss': 'o'}
DODGE   = {'ser': -0.10, 'tloss': 0.10}
LN2 = np.log(2.0)

CARGO_INCLUDE_REGEX = (r'bulk|cellular|container|general cargo|roll on roll off|ro-ro|roro|reefer|'
                       r'vehicle carrier|livestock|wood-chip|barge carrier|ore carrier|aggregates|'
                       r'combined|cargo ship')
CARGO_EXCLUDE_REGEX = (r'passenger|ferry|cruise|tanker|fishing|trawler|tug|supply|offshore|'
                       r'dredg|yacht|patrol')

def normalise_text(s): return s.astype('string').str.strip().str.lower()

def classify_cargo_subtype(vessel_type):
    vt = normalise_text(vessel_type)
    conditions = [
        vt.str.contains(r'bulk|ore carrier|wood-chip|aggregates', regex=True, na=False),
        vt.str.contains(r'container|cellular', regex=True, na=False),
        vt.str.contains(r'roll on roll off|ro-ro|roro|vehicle carrier', regex=True, na=False),
        vt.str.contains(r'reefer|refrigerated', regex=True, na=False),
        vt.str.contains(r'general cargo|cargo ship|combined|barge carrier', regex=True, na=False),
    ]
    choices = ['bulk', 'container', 'ro_ro_vehicle', 'reefer', 'general_other']
    return pd.Series(np.select(conditions, choices, default='other_cargo'),
                     index=vessel_type.index, dtype='string')

def build_analysis_sample(path):
    raw = pd.read_parquet(path)
    audit = {'n_raw': int(len(raw))}
    vt = normalise_text(raw['vessel_type'])
    is_cargo = (vt.str.contains(CARGO_INCLUDE_REGEX, regex=True, na=False)
                & ~vt.str.contains(CARGO_EXCLUDE_REGEX, regex=True, na=False))
    df = raw.loc[is_cargo].copy()
    audit['n_cargo_regex'] = int(len(df))

    serious_map = {'true': 1, '1': 1, '1.0': 1, 'yes': 1, 'y': 1,
                   'false': 0, '0': 0, '0.0': 0, 'no': 0, 'n': 0}
    df['ser'] = normalise_text(df['serious']).map(serious_map).astype('Float64')
    loss_norm = normalise_text(df['loss_type'])
    df['tloss'] = loss_norm.isin(TOTAL_LOSS_CATEGORIES).fillna(False).astype('int8')
    audit['n_actual_total_loss'] = int(loss_norm.eq('actual total loss').fillna(False).sum())
    audit['n_constructive_total_loss'] = int(loss_norm.eq('constructive total loss').fillna(False).sum())

    df['date'] = pd.to_datetime(df['casualty_date'], errors='coerce')
    for col in ['swh_daily_max', 'wind_daily_max', 'age', 'GT', 'lat', 'lon']:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    ok = {
        'swh_missing':        df['swh_daily_max'].ge(0),
        'wind_missing':       df['wind_daily_max'].ge(0),
        'age_missing_or_gt60': df['age'].between(0, 60, inclusive='both'),
        'gt_zero_or_missing': df['GT'].gt(0),
        'date_missing':       df['date'].notna(),
        'latlon_invalid':     df['lat'].between(-90, 90) & df['lon'].between(-180, 180),
        'location_missing':   df['casualty_location'].notna(),
        'serious_unmapped':   df['ser'].notna(),
    }
    valid_core = np.logical_and.reduce([m.fillna(False).to_numpy() for m in ok.values()])
    for k, m in ok.items():
        audit[f'n_excl_{k}'] = int((~m.fillna(False)).sum())
    others_ok = np.logical_and.reduce([m.fillna(False).to_numpy() for k, m in ok.items() if k != 'swh_missing'])
    audit['n_excl_only_swh_missing'] = int((~ok['swh_missing'].fillna(False).to_numpy() & others_ok).sum())
    audit['n_invalid_core'] = int((~valid_core).sum())
    df = df.loc[valid_core].copy()
    df['ser'] = df['ser'].astype(int); df['tloss'] = df['tloss'].astype(int)

    df['wv'] = pd.cut(df['swh_daily_max'], WAVE_EDGES, right=False, labels=WAVE_LABELS, ordered=True)
    df['wd'] = pd.cut(df['wind_daily_max'], WIND_EDGES, right=False, labels=WIND_LABELS, ordered=True)
    assert not df[['wv', 'wd']].isna().any().any()

    df['yr'] = df['date'].dt.year.astype(int).astype(str)
    df['mo'] = df['date'].dt.month.astype(int).astype(str)
    df['area'] = normalise_text(df['casualty_location'])
    df['area_month'] = df['area'] + '_m' + df['mo']
    lat5, lon5   = np.floor(df['lat']/5).astype(int)*5,  np.floor(df['lon']/5).astype(int)*5
    lat10, lon10 = np.floor(df['lat']/10).astype(int)*10, np.floor(df['lon']/10).astype(int)*10
    df['g5']  = lat5.astype(str) + '_' + lon5.astype(str)
    df['g10'] = lat10.astype(str) + '_' + lon10.astype(str)
    df['g10_month'] = df['g10'] + '_m' + df['mo']
    df['cargo_subtype'] = classify_cargo_subtype(df['vessel_type'])

    age_ref, gt_ref = float(df['age'].median()), float(df['GT'].median())
    df['age5c'] = (df['age'] - age_ref) / 5.0
    df['lgtc']  = np.log(df['GT'] / gt_ref)
    df['age5c_sq'] = df['age5c']**2; df['lgtc_sq'] = df['lgtc']**2
    for c in ['yr','mo','area','area_month','g5','g10','g10_month','cargo_subtype']:
        df[c] = df[c].astype(object)

    audit.update(n_analysis=int(len(df)), age_reference_years=age_ref, gt_reference=gt_ref,
                 n_clusters_g5=int(df['g5'].nunique()), n_g10=int(df['g10'].nunique()),
                 n_areas=int(df['area'].nunique()))
    return df, audit

g, SAMPLE_AUDIT = build_analysis_sample(ACC_FP)

print('Sample audit'); print(pd.Series(SAMPLE_AUDIT).to_string())
print('\nOutcome prevalence')
for y, lab in OUTCOMES.items():
    print(f'  {lab:16s} {g[y].sum():>6,}/{len(g):,}  ({100*g[y].mean():.2f}%)')
for hz, meta in HAZARDS.items():
    t = g.groupby(hz, observed=True).agg(n=('ser','size'), ser=('ser','sum'), tloss=('tloss','sum'))
    print(f'\n{meta["name"]}'); print(t.to_string())
print('\nCargo subtypes'); print(g['cargo_subtype'].value_counts().to_string())


In [ ]:
FE_CHOICES = {'area_month': 'C(area_month) + C(yr)',
              'g10':        'C(g10) + C(mo) + C(yr)',
              'g10_month':  'C(g10_month) + C(yr)'}
FE_RHS = FE_CHOICES[FE_MODE]
SUBTYPE_RHS = ' + C(cargo_subtype)' if USE_CARGO_SUBTYPE_CONTROL else ''

def cterm(var, ref): return f'C({var}, Treatment(reference="{ref}"))'
WV, WD = cterm('wv', WAVE_LABELS[0]), cterm('wd', WIND_LABELS[0])

def fit_lpm(df, outcome, rhs):
    formula = f'{outcome} ~ {rhs} + {FE_RHS}'
    res = smf.ols(formula, data=df, missing='raise').fit(
        cov_type='cluster', cov_kwds={'groups': df[CLUSTER_COL], 'use_correction': True})
    res._m3_formula = formula
    return res

def fit_binomial_glm(df, outcome, rhs):
    formula = f'{outcome} ~ {rhs} + {FE_RHS}'
    res = smf.glm(formula, data=df, family=sm.families.Binomial(), missing='raise').fit(
        cov_type='cluster', cov_kwds={'groups': df[CLUSTER_COL], 'use_correction': True}, maxiter=200)
    res._m3_formula = formula
    return res

def _tok(label): return f'[T.{label}]'
def find_main_term(res, var, label):
    c = [n for n in res.params.index if n.startswith(f'C({var}') and _tok(label) in n and ':' not in n]
    assert len(c) == 1, c; return c[0]
def find_interaction_term(res, var, label, mod):
    c = [n for n in res.params.index if n.startswith(f'C({var}') and _tok(label) in n
         and ':' in n and mod in n.split(':')]
    assert len(c) == 1, c; return c[0]
def interaction_terms(res, var, mod):
    return [n for n in res.params.index if n.startswith(f'C({var}') and ':' in n and mod in n.split(':')]
def main_category_terms(res, var):
    return [n for n in res.params.index if n.startswith(f'C({var}') and ':' not in n]

def joint_pvalue(res, terms):
    if not terms: return np.nan
    names = list(res.params.index); R = np.zeros((len(terms), len(names)))
    for i, t in enumerate(terms): R[i, names.index(t)] = 1.0
    return float(np.asarray(res.f_test(R).pvalue).squeeze())

def linear_contrast(res, weights, scale=1.0):
    names = list(res.params.index); v = np.zeros(len(names))
    for t, w in weights.items(): v[names.index(t)] = w
    b = float(v @ np.asarray(res.params)) * scale
    var = float(v @ np.asarray(res.cov_params()) @ v) * scale**2
    se = np.sqrt(max(var, 0.0)); z = b/se if se > 0 else np.nan
    p = float(2*norm.sf(abs(z))) if np.isfinite(z) else np.nan
    return dict(b=b, se=se, lo=b-1.96*se, hi=b+1.96*se, p=p)

def p_text(p):
    if not np.isfinite(p): return 'NA'
    return '<0.001' if p < 0.001 else f'{p:.3f}'

print('Fixed effects:', FE_RHS)
print('Cluster variable:', CLUSTER_COL, f'({g[CLUSTER_COL].nunique()} clusters)')
print('Subtype control:', SUBTYPE_RHS or 'none')


In [ ]:
main_models, MAIN_RHS = {}, {}
for y in OUTCOMES:
    for hz, meta in HAZARDS.items():
        rhs = f'{cterm(hz, meta["reference"])} + age5c + lgtc{SUBTYPE_RHS}'
        MAIN_RHS[hz] = rhs
        main_models[(y, hz)] = fit_lpm(g, y, rhs)

def build_main_effect_table(models, jointly_adjusted=False):
    rows = []
    for y, ylab in OUTCOMES.items():
        for hz, meta in HAZARDS.items():
            res = models[y] if jointly_adjusted else models[(y, hz)]
            pj = joint_pvalue(res, main_category_terms(res, hz))
            for i, lab in enumerate(meta['labels']):
                eff = (dict(b=0.0, se=0.0, lo=0.0, hi=0.0, p=np.nan) if i == 0
                       else linear_contrast(res, {find_main_term(res, hz, lab): 1.0}, scale=100.0))
                rows.append(dict(outcome=y, outcome_label=ylab, hazard=hz, category=lab,
                                 category_index=i, **eff, joint_p=pj, n=int(res.nobs)))
    return pd.DataFrame(rows)

T_MAIN = build_main_effect_table(main_models)
T_MAIN.to_csv(OUT_DIR / 'table_main_separate_hazards.csv', index=False)
print('Dose-response by hazard (risk difference, pp)')
print(T_MAIN[['outcome','hazard','category','b','lo','hi','p','joint_p','n']].round(3).to_string(index=False))


In [ ]:
JOINT_RHS = f'{WV} + {WD} + age5c + lgtc{SUBTYPE_RHS}'
joint_models = {y: fit_lpm(g, y, JOINT_RHS) for y in OUTCOMES}
T_JOINT = build_main_effect_table(joint_models, jointly_adjusted=True)
T_JOINT.to_csv(OUT_DIR / 'table_sensitivity_joint_weather.csv', index=False)
print('Waves and wind mutually adjusted')
print(T_JOINT[['outcome','hazard','category','b','lo','hi','p','joint_p']].round(3).to_string(index=False))

rows = []
for y, ylab in OUTCOMES.items():
    for hz, meta in HAZARDS.items():
        cat = meta['labels'][-1]
        for spec, res in [('separate_hazard_adjusted', main_models[(y, hz)]),
                          ('mutually_adjusted', joint_models[y])]:
            eff = linear_contrast(res, {find_main_term(res, hz, cat): 1.0}, scale=100.0)
            rows.append(dict(outcome=y, hazard=hz, category=cat, specification=spec, **eff, n=int(res.nobs)))
T_HIGH_COMPARISON = pd.DataFrame(rows)
T_HIGH_COMPARISON.to_csv(OUT_DIR / 'table_highest_category_model_comparison.csv', index=False)
print('\nTop bin: alone vs mutually adjusted')
print(T_HIGH_COMPARISON.round(3).to_string(index=False))


In [ ]:
def build_modifier_table(models, modifier, scale):
    rows = []
    for y, ylab in OUTCOMES.items():
        for hz, meta in HAZARDS.items():
            res = models[(y, hz)]
            pj = joint_pvalue(res, interaction_terms(res, hz, modifier))
            for i, lab in enumerate(meta['labels']):
                w = {modifier: 1.0}
                if i == 0:
                    inter = dict(b=0.0, se=0.0, lo=0.0, hi=0.0, p=np.nan)
                else:
                    it = find_interaction_term(res, hz, lab, modifier); w[it] = 1.0
                    inter = linear_contrast(res, {it: 1.0}, scale=scale)
                slope = linear_contrast(res, w, scale=scale)
                rows.append(dict(outcome=y, outcome_label=ylab, hazard=hz, category=lab, category_index=i,
                                 **{f'slope_{k}': v for k, v in slope.items()},
                                 **{f'interaction_{k}': v for k, v in inter.items()},
                                 joint_interaction_p=pj, n=int(res.nobs)))
    return pd.DataFrame(rows)

age_models, AGE_RHS = {}, {}
for y in OUTCOMES:
    for hz, meta in HAZARDS.items():
        rhs = f'{cterm(hz, meta["reference"])} * age5c + lgtc{SUBTYPE_RHS}'
        AGE_RHS[hz] = rhs
        age_models[(y, hz)] = fit_lpm(g, y, rhs)

T_AGE = build_modifier_table(age_models, 'age5c', 100.0)
T_AGE.to_csv(OUT_DIR / 'table_age_interactions.csv', index=False)
print('Vessel-age slope (pp / 5 yr) and interaction contrasts')
print(T_AGE[['outcome','hazard','category','slope_b','slope_lo','slope_hi',
             'interaction_b','interaction_lo','interaction_hi','interaction_p','joint_interaction_p']]
      .round(3).to_string(index=False))


In [ ]:
size_models, SIZE_RHS = {}, {}
for y in OUTCOMES:
    for hz, meta in HAZARDS.items():
        rhs = f'{cterm(hz, meta["reference"])} * lgtc + age5c{SUBTYPE_RHS}'
        SIZE_RHS[hz] = rhs
        size_models[(y, hz)] = fit_lpm(g, y, rhs)

T_SIZE = build_modifier_table(size_models, 'lgtc', 100.0 * LN2)
T_SIZE.to_csv(OUT_DIR / 'table_size_interactions.csv', index=False)
print('Size slope (pp / GT doubling) and interaction contrasts')
print(T_SIZE[['outcome','hazard','category','slope_b','slope_lo','slope_hi',
              'interaction_b','interaction_lo','interaction_hi','interaction_p','joint_interaction_p']]
      .round(3).to_string(index=False))


In [ ]:
def build_weather_contrasts_at_values(models, modifier, values):
    rows = []
    for y, ylab in OUTCOMES.items():
        for hz, meta in HAZARDS.items():
            res = models[(y, hz)]
            for i, cat in enumerate(meta['labels']):
                for vlab, orig, cen in values:
                    if i == 0:
                        eff = dict(b=0.0, se=0.0, lo=0.0, hi=0.0, p=np.nan)
                    else:
                        eff = linear_contrast(res, {find_main_term(res, hz, cat): 1.0,
                                                    find_interaction_term(res, hz, cat, modifier): cen}, scale=100.0)
                    rows.append(dict(outcome=y, outcome_label=ylab, hazard=hz, category=cat, category_index=i,
                                     modifier=modifier, value_label=vlab, original_value=orig,
                                     centred_model_value=cen, **eff, n=int(res.nobs)))
    return pd.DataFrame(rows)

age_ref = SAMPLE_AUDIT['age_reference_years']
age_values = [(f'Age {a:.0f} years', a, (a - age_ref)/5.0) for a in [5.0, 15.0, 25.0, 35.0]]
T_WEATHER_BY_AGE = build_weather_contrasts_at_values(age_models, 'age5c', age_values)
T_WEATHER_BY_AGE.to_csv(OUT_DIR / 'table_weather_effects_at_representative_ages.csv', index=False)

gt_ref = SAMPLE_AUDIT['gt_reference']
gt_values = [(f'GT p{int(q*100):02d}', float(v), float(np.log(v/gt_ref)))
             for q, v in g['GT'].quantile([0.10, 0.50, 0.90]).items()]
T_WEATHER_BY_SIZE = build_weather_contrasts_at_values(size_models, 'lgtc', gt_values)
T_WEATHER_BY_SIZE.to_csv(OUT_DIR / 'table_weather_effects_at_representative_GT.csv', index=False)

cols = ['outcome','hazard','category','value_label','b','lo','hi','p']
print('Weather effects at representative vessel ages (pp)'); print(T_WEATHER_BY_AGE[cols].round(2).to_string(index=False))
print('\nWeather effects at representative GT (pp)'); print(T_WEATHER_BY_SIZE[cols].round(2).to_string(index=False))


In [ ]:
full_models, rows = {}, []
if RUN_FULL_INTERACTION:
    for y, ylab in OUTCOMES.items():
        for hz, meta in HAZARDS.items():
            ht = cterm(hz, meta['reference'])
            res = fit_lpm(g, y, f'{ht} * age5c + {ht} * lgtc{SUBTYPE_RHS}')
            full_models[(y, hz)] = res
            for mod in ['age5c', 'lgtc']:
                terms = interaction_terms(res, hz, mod)
                rows.append(dict(outcome=y, hazard=hz, modifier=mod,
                                 joint_interaction_p=joint_pvalue(res, terms), n_terms=len(terms), n=int(res.nobs)))
    T_FULL_TESTS = pd.DataFrame(rows)
    T_FULL_TESTS.to_csv(OUT_DIR / 'table_full_model_interaction_tests.csv', index=False)
    print('Joint tests, full interaction model'); print(T_FULL_TESTS.round(4).to_string(index=False))

if RUN_NONLINEAR:
    rows = []
    for y, ylab in OUTCOMES.items():
        for hz, meta in HAZARDS.items():
            ht = cterm(hz, meta['reference'])
            for mod, sq, other in [('age5c', 'age5c_sq', 'lgtc'), ('lgtc', 'lgtc_sq', 'age5c')]:
                res = fit_lpm(g, y, f'{ht} * ({mod} + {sq}) + {other}{SUBTYPE_RHS}')
                terms = interaction_terms(res, hz, mod) + interaction_terms(res, hz, sq)
                rows.append(dict(outcome=y, hazard=hz, modifier=mod, specification='linear_plus_quadratic',
                                 joint_interaction_p=joint_pvalue(res, terms), n_terms=len(terms)))
    T_NONLINEAR = pd.DataFrame(rows)
    T_NONLINEAR.to_csv(OUT_DIR / 'table_nonlinear_interaction_tests.csv', index=False)
    print('\nNonlinear interaction tests'); print(T_NONLINEAR.round(4).to_string(index=False))

if RUN_GLM_SENSITIVITY:
    rows = []
    for y, ylab in OUTCOMES.items():
        for hz in HAZARDS:
            try:
                res = fit_binomial_glm(g, y, MAIN_RHS[hz])
                rows.append(dict(outcome=y, hazard=hz, joint_p=joint_pvalue(res, main_category_terms(res, hz)),
                                 converged=bool(getattr(res, 'converged', True)), n=int(res.nobs)))
            except Exception as e:
                print(f'  GLM failed {y}/{hz}: {type(e).__name__}: {e}')
                rows.append(dict(outcome=y, hazard=hz, joint_p=np.nan, converged=False, n=len(g)))
    T_GLM = pd.DataFrame(rows)
    T_GLM.to_csv(OUT_DIR / 'table_glm_sensitivity.csv', index=False)
    print('\nBinomial GLM sensitivity'); print(T_GLM.round(4).to_string(index=False))


In [ ]:
INK, GREY, LIGHT_GREY, SPINE = '#1a1a1a', '#72777d', '#f3f4f5', '#c8cdd2'

def colour_for(hz, y): return HAZARDS[hz]['colour_light'] if y == 'ser' else HAZARDS[hz]['colour_dark']

def style_axis(ax):
    for s in ('top', 'right'): ax.spines[s].set_visible(False)
    for s in ('left', 'bottom'): ax.spines[s].set_color(SPINE)
    ax.tick_params(colors=GREY, labelsize=8.5); ax.set_axisbelow(True)

def plot_dose_response(ax, hz, panel_label, annot=True):
    meta = HAZARDS[hz]; labels = meta['labels']; x = np.arange(len(labels))
    for i in range(0, len(labels), 2): ax.axvspan(i-.5, i+.5, color=LIGHT_GREY, lw=0, zorder=0)
    joint_lines = []
    for y, ylab in OUTCOMES.items():
        d = T_MAIN[(T_MAIN.hazard == hz) & (T_MAIN.outcome == y)].sort_values('category_index')
        col = colour_for(hz, y); xx = x + DODGE[y]
        ax.plot(xx, d['b'], color=col, lw=1.4, zorder=2)
        ax.errorbar(xx[1:], d['b'].iloc[1:], yerr=1.96*d['se'].iloc[1:], fmt=MARKERS[y], ms=5.3,
                    mfc=col, mec=col, ecolor=col, elinewidth=1.1, capsize=2.5, zorder=3)
        if annot:
            for xi, r in zip(xx[1:], d.iloc[1:].itertuples()):
                off = 0.7 if y == 'ser' else -0.9
                ax.text(xi, r.b + off, f'{r.b:.2f}', ha='center',
                        va='bottom' if y == 'ser' else 'top', fontsize=7.2, color=col)
        joint_lines.append(f'{ylab}: $P_{{joint}}$ {p_text(d["joint_p"].iloc[0])}')
    ax.plot(0, 0, marker='D', ms=5.2, mfc='white', mec=GREY, mew=1.2, zorder=5)
    ax.axhline(0, color=GREY, lw=.8, ls=(0, (4, 3)))
    ax.set_xticks(x); ax.set_xticklabels([l.replace('>=', '≥') for l in labels], fontsize=8.4)
    ax.set_xlim(-.55, len(labels)-.45)
    ax.set_xlabel(meta['symbol'], fontsize=9.2, color=INK)
    ax.set_ylabel('Adjusted probability difference (pp)', fontsize=9.2, color=INK)
    ax.grid(axis='y', color='#e5e8ea', lw=.5)
    if annot:
        ax.text(.02, .97, '\n'.join(joint_lines), transform=ax.transAxes, ha='left', va='top', fontsize=7.5, color=INK)
        ax.text(-.12, 1.08, panel_label, transform=ax.transAxes, fontsize=13, fontweight='bold', color=INK)
        ax.set_title(meta['name'], fontsize=10.2, loc='left', pad=8, color=INK)
    style_axis(ax)

def finite_limits(lo, hi):
    l = float(np.nanmin(np.r_[lo.to_numpy(), [0.]])); h = float(np.nanmax(np.r_[hi.to_numpy(), [0.]]))
    span = max(h - l, 1.0); return l - .12*span, h + .12*span

def plot_interaction_contrasts(ax, table, hz, modifier, xlim, panel_label=None, annot=True):
    meta = HAZARDS[hz]; labels = meta['labels']; n = len(labels); yv = np.arange(n)
    for i in range(0, n, 2): ax.axhspan(i-.5, i+.5, color=LIGHT_GREY, lw=0, zorder=0)
    joint_lines = []
    for y, ylab in OUTCOMES.items():
        d = table[(table.hazard == hz) & (table.outcome == y)].sort_values('category_index')
        col = colour_for(hz, y); yy = yv + DODGE[y]
        for yp, r in zip(yy, d.itertuples()):
            if r.category_index == 0:
                ax.plot(0, yp, marker=MARKERS[y], ms=5, mfc='white', mec=col, mew=1.1, zorder=3); continue
            sig = np.isfinite(r.interaction_p) and r.interaction_p < .05
            ax.plot([r.interaction_lo, r.interaction_hi], [yp, yp], color=col, lw=1.1, alpha=.78, zorder=1)
            ax.plot(r.interaction_b, yp, marker=MARKERS[y], ms=5, mfc=col if sig else 'white', mec=col, mew=1.1, zorder=3)
        joint_lines.append(f'{ylab}: $P_{{interaction}}$ {p_text(d["joint_interaction_p"].iloc[0])}')
    ax.axvline(0, color=GREY, lw=.8, ls=(0, (4, 3)))
    ax.set_yticks(yv); ax.set_yticklabels([l.replace('>=', '≥') for l in labels], fontsize=8.2)
    ax.invert_yaxis(); ax.set_xlim(*xlim); ax.grid(axis='x', color='#e5e8ea', lw=.5)
    if modifier == 'age':
        ttl, xl = 'Age interaction', 'Difference in age gradient\n(pp per 5 years vs reference)'
    else:
        ttl, xl = 'Size interaction', 'Difference in size gradient\n(pp per GT doubling vs reference)'
    ax.set_xlabel(xl, fontsize=8.3, color=INK)
    if annot:
        ax.text(.5, 1.015, '   '.join(joint_lines), transform=ax.transAxes, ha='center', va='bottom', fontsize=6.6, color=INK)
        ax.set_title(ttl, fontsize=9.2, color=INK, pad=25)
        if panel_label:
            ax.text(-.22, 1.10, panel_label, transform=ax.transAxes, fontsize=13, fontweight='bold', color=INK)
    style_axis(ax); ax.spines['left'].set_visible(False); ax.tick_params(axis='y', length=0)

DOSE_LEGEND = [Line2D([], [], marker='s', ls='', ms=5.5, mfc='#6f8794', mec='#6f8794', label='Serious outcome'),
               Line2D([], [], marker='o', ls='', ms=5.5, mfc='#344c59', mec='#344c59', label='Total loss'),
               Line2D([], [], marker='D', ls='', ms=5.0, mfc='white', mec=GREY, label='Reference')]
INTER_LEGEND = DOSE_LEGEND[:2] + [Line2D([], [], marker='o', ls='', ms=5.0, mfc='white', mec=GREY,
                                         label='Interaction 95% CI includes 0')]
AGE_XLIM  = finite_limits(T_AGE['interaction_lo'],  T_AGE['interaction_hi'])
SIZE_XLIM = finite_limits(T_SIZE['interaction_lo'], T_SIZE['interaction_hi'])

def show_dose(hz, panel_label, annot):
    fig, ax = plt.subplots(figsize=(6.4, 4.8), dpi=150)
    plot_dose_response(ax, hz, panel_label, annot)
    if annot:
        fig.legend(handles=DOSE_LEGEND, loc='lower center', bbox_to_anchor=(.5, .015), ncol=3, frameon=False, fontsize=8)
    fig.subplots_adjust(left=.14, right=.98, top=.88, bottom=.22 if annot else .14)
    plt.show()

def show_inter(table, hz, modifier, xlim, panel_label, annot):
    fig, ax = plt.subplots(figsize=(5.4, 4.8), dpi=150)
    plot_interaction_contrasts(ax, table, hz, modifier, xlim, panel_label, annot)
    if annot:
        fig.legend(handles=INTER_LEGEND, loc='lower center', bbox_to_anchor=(.5, .015), ncol=3, frameon=False, fontsize=7.4)
    fig.subplots_adjust(left=.20, right=.97, top=.84 if annot else .95, bottom=.24 if annot else .18)
    plt.show()

PANELS = [('dose',  'wv', None,   'a'),  ('inter', 'wv', 'age',  'b1'), ('inter', 'wv', 'size', 'b2'),
          ('dose',  'wd', None,   'c'),  ('inter', 'wd', 'age',  'd1'), ('inter', 'wd', 'size', 'd2')]

def show_all(annot):
    for kind, hz, mod, lab in PANELS:
        if kind == 'dose': show_dose(hz, lab, annot)
        elif mod == 'age': show_inter(T_AGE,  hz, 'age',  AGE_XLIM,  lab, annot)
        else:              show_inter(T_SIZE, hz, 'size', SIZE_XLIM, lab, annot)


In [ ]:
show_all(annot=True)


In [ ]:
show_all(annot=False)


In [ ]:
SLOPE_XLIM_AGE  = finite_limits(T_AGE['slope_lo'],  T_AGE['slope_hi'])
SLOPE_XLIM_SIZE = finite_limits(T_SIZE['slope_lo'], T_SIZE['slope_hi'])

def plot_modifier_slopes(ax, table, hz, modifier, xlim, panel_label=None, annot=True):
    meta = HAZARDS[hz]; labels = meta['labels']; n = len(labels); yv = np.arange(n)
    for i in range(0, n, 2): ax.axhspan(i-.5, i+.5, color=LIGHT_GREY, lw=0, zorder=0)
    joint_lines = []
    for y, ylab in OUTCOMES.items():
        d = table[(table.hazard == hz) & (table.outcome == y)].sort_values('category_index')
        col = colour_for(hz, y); yy = yv + DODGE[y]
        for yp, r in zip(yy, d.itertuples()):
            sig = np.isfinite(r.slope_p) and r.slope_p < .05
            ax.plot([r.slope_lo, r.slope_hi], [yp, yp], color=col, lw=1.1, alpha=.78, zorder=1)
            ax.plot(r.slope_b, yp, marker=MARKERS[y], ms=5, mfc=col if sig else 'white',
                    mec=col, mew=1.1, zorder=3)
            if annot:
                ax.text(r.slope_hi + .04*(xlim[1]-xlim[0]), yp, f'{r.slope_b:.2f}',
                        va='center', ha='left', fontsize=6.8, color=col)
        joint_lines.append(f'{ylab}: $P_{{interaction}}$ {p_text(d["joint_interaction_p"].iloc[0])}')
    ax.axvline(0, color=GREY, lw=.8, ls=(0, (4, 3)))
    ax.set_yticks(yv); ax.set_yticklabels([l.replace('>=', '≥') for l in labels], fontsize=8.2)
    ax.invert_yaxis(); ax.set_xlim(*xlim); ax.grid(axis='x', color='#e5e8ea', lw=.5)
    if modifier == 'age':
        ttl, xl = 'Age gradient by sea state', 'Change in outcome probability\n(pp per 5 years of vessel age)'
    else:
        ttl, xl = 'Size gradient by sea state', 'Change in outcome probability\n(pp per doubling of GT)'
    ax.set_xlabel(xl, fontsize=8.3, color=INK)
    if annot:
        ax.text(.5, 1.015, '   '.join(joint_lines), transform=ax.transAxes,
                ha='center', va='bottom', fontsize=6.6, color=INK)
        ax.set_title(ttl, fontsize=9.2, color=INK, pad=25)
        if panel_label:
            ax.text(-.22, 1.10, panel_label, transform=ax.transAxes, fontsize=13, fontweight='bold', color=INK)
    style_axis(ax); ax.spines['left'].set_visible(False); ax.tick_params(axis='y', length=0)

SLOPE_LEGEND = DOSE_LEGEND[:2] + [Line2D([], [], marker='o', ls='', ms=5.0, mfc='white', mec=GREY,
                                         label='Slope 95% CI includes 0')]

def show_slopes(table, hz, modifier, xlim, panel_label, annot):
    fig, ax = plt.subplots(figsize=(5.4, 4.8), dpi=150)
    plot_modifier_slopes(ax, table, hz, modifier, xlim, panel_label, annot)
    if annot:
        fig.legend(handles=SLOPE_LEGEND, loc='lower center', bbox_to_anchor=(.5, .015),
                   ncol=3, frameon=False, fontsize=7.4)
    fig.subplots_adjust(left=.20, right=.97, top=.84 if annot else .95, bottom=.24 if annot else .18)
    plt.show()

# Vessel-age slope: waves (b1') and wind (d1')
for annot in (True, False):
    show_slopes(T_AGE, 'wv', 'age', SLOPE_XLIM_AGE, "b1'", annot)
    show_slopes(T_AGE, 'wd', 'age', SLOPE_XLIM_AGE, "d1'", annot)


In [ ]:
def cell_diff(res, cf1, cf2, scale=100.0):
    """Difference in mean predictions between two counterfactual samples (pp), CI, p"""
    di = res.model.data.design_info
    x1 = np.asarray(build_design_matrices([di], cf1)[0]).mean(axis=0)
    x2 = np.asarray(build_design_matrices([di], cf2)[0]).mean(axis=0)
    v = x1 - x2
    b = float(v @ res.params.values) * scale
    se = float(np.sqrt(v @ res.cov_params().values @ v)) * scale
    z = b / se if se > 0 else np.nan
    return dict(b=b, se=se, lo=b-1.96*se, hi=b+1.96*se, p=float(2*norm.sf(abs(z))))

def make_cf(hz, lab, age):
    cf = g.copy()
    cf[hz] = pd.Categorical([lab]*len(cf), categories=HAZARDS[hz]['labels'], ordered=True)
    cf['age5c'] = (age - age_ref) / 5.0
    return cf

rows = []
for hz, meta in HAZARDS.items():
    labels = meta['labels']
    for y, ylab in OUTCOMES.items():
        res = age_models[(y, hz)]
        for lab in labels:                                   # (1) same wave bin: age 35 vs 5
            rows.append(dict(hazard=hz, outcome=y, contrast='age 35 vs 5', category=lab, age=np.nan,
                             **cell_diff(res, make_cf(hz, lab, 35), make_cf(hz, lab, 5))))
        for a in AGE_GRID:                                   # (2) same vessel age: top bin vs reference
            rows.append(dict(hazard=hz, outcome=y, contrast=f'{labels[-1]} vs {labels[0]}', category=labels[-1], age=a,
                             **cell_diff(res, make_cf(hz, labels[-1], a), make_cf(hz, labels[0], a))))
        rows.append(dict(hazard=hz, outcome=y, contrast='corner: top bin age 35 vs ref bin age 5',  # (3) diagonal
                         category=labels[-1], age=35,
                         **cell_diff(res, make_cf(hz, labels[-1], 35), make_cf(hz, labels[0], 5))))
T_PRED_DIFF = pd.DataFrame(rows)
T_PRED_DIFF.to_csv(OUT_DIR / 'table_predicted_probability_contrasts.csv', index=False)
print(T_PRED_DIFF.round(3).to_string(index=False))


In [ ]:
def show_pred_heatmap(hz, y, panel_label=None, annot=True):
    meta = HAZARDS[hz]; labels = meta['labels']
    d = T_PRED_AGE[(T_PRED_AGE.hazard == hz) & (T_PRED_AGE.outcome == y)]
    M  = d.pivot(index='category', columns='age', values='b').reindex(labels)
    LO = d.pivot(index='category', columns='age', values='lo').reindex(labels)
    HI = d.pivot(index='category', columns='age', values='hi').reindex(labels)
    ages = M.columns.to_numpy()
    fig, ax = plt.subplots(figsize=(7.2, 3.9), dpi=150)
    im = ax.imshow(M.values, cmap='Blues' if hz == 'wv' else 'Oranges', aspect='auto',
                   vmin=0, vmax=np.nanmax(M.values))
    ax.set_xticks(range(len(ages))); ax.set_xticklabels([f'{a:g}' for a in ages], fontsize=8.5)
    ax.set_yticks(range(len(labels))); ax.set_yticklabels([l.replace('>=', '≥') for l in labels], fontsize=8.5)
    ax.set_xlabel('Vessel age (years)', fontsize=9.2, color=INK)
    ax.set_ylabel(meta['symbol'], fontsize=9.2, color=INK)
    ax.tick_params(length=0, colors=GREY)
    for s in ax.spines.values(): s.set_visible(False)
    thr = 0.6 * np.nanmax(M.values)
    for i in range(len(labels)):
        for j in range(len(ages)):
            v = M.values[i, j]; tc = 'white' if v > thr else INK
            if annot:
                ax.text(j, i - .12, f'{v:.1f}', ha='center', va='center', fontsize=8.6, fontweight='bold', color=tc)
                ax.text(j, i + .22, f'({LO.values[i,j]:.1f}–{HI.values[i,j]:.1f})',
                        ha='center', va='center', fontsize=6.4, color=tc)
            else:
                ax.text(j, i, f'{v:.1f}', ha='center', va='center', fontsize=8.6, color=tc)
    cb = fig.colorbar(im, ax=ax, fraction=.035, pad=.02); cb.ax.tick_params(labelsize=7.5, colors=GREY)
    cb.outline.set_visible(False)
    if annot:
        cb.set_label(f'Predicted probability of {OUTCOMES[y].lower()} (%)', fontsize=8, color=INK)
        pj = T_AGE[(T_AGE.hazard == hz) & (T_AGE.outcome == y)]['joint_interaction_p'].iloc[0]
        ax.set_title(f'{meta["name"]} × vessel age     $P_{{interaction}}$ {p_text(pj)}',
                     fontsize=9.6, loc='left', pad=8, color=INK)
        if panel_label:
            ax.text(-.14, 1.08, panel_label, transform=ax.transAxes, fontsize=13, fontweight='bold', color=INK)
    fig.tight_layout(); plt.show()

for annot in (True, False):
    show_pred_heatmap('wv', 'tloss', 'a', annot)
    show_pred_heatmap('wv', 'ser',   'b', annot)
    show_pred_heatmap('wd', 'tloss', 'c', annot)
    show_pred_heatmap('wd', 'ser',   'd', annot)


In [ ]:
AGE_GRID = np.arange(5, 36, 5)                              # 5 … 35
GT_GRID  = np.array([1000, 2000, 4000, 8000, 16000, 32000, 64000])   # doubles each column
age_ref, gt_ref = SAMPLE_AUDIT['age_reference_years'], SAMPLE_AUDIT['gt_reference']

def make_cf(hz, lab, modifier, value):
    cf = g.copy()
    cf[hz] = pd.Categorical([lab]*len(cf), categories=HAZARDS[hz]['labels'], ordered=True)
    if modifier == 'age': cf['age5c'] = (value - age_ref) / 5.0
    else:                 cf['lgtc']  = np.log(value / gt_ref)
    return cf

def predicted_table(models, modifier, grid):
    rows = []
    for hz, meta in HAZARDS.items():
        for y in OUTCOMES:
            res = models[(y, hz)]
            for lab in meta['labels']:
                for v in grid:
                    rows.append(dict(hazard=hz, outcome=y, category=lab, value=v,
                                     **avg_prediction(res, make_cf(hz, lab, modifier, v))))
    return pd.DataFrame(rows)

T_PRED_AGE  = predicted_table(age_models,  'age',  AGE_GRID)
T_PRED_SIZE = predicted_table(size_models, 'size', GT_GRID)
T_PRED_AGE.to_csv(OUT_DIR / 'table_predicted_by_age.csv', index=False)
T_PRED_SIZE.to_csv(OUT_DIR / 'table_predicted_by_GT.csv', index=False)

def _heat(ax, fig, table, hz, y, modifier, annot):
    meta = HAZARDS[hz]; labels = meta['labels']
    d = table[(table.hazard == hz) & (table.outcome == y)]
    M  = d.pivot(index='category', columns='value', values='b').reindex(labels)
    LO = d.pivot(index='category', columns='value', values='lo').reindex(labels)
    HI = d.pivot(index='category', columns='value', values='hi').reindex(labels)
    vals = M.columns.to_numpy()
    im = ax.imshow(M.values, cmap='Blues' if hz == 'wv' else 'Oranges', aspect='auto',
                   vmin=0, vmax=np.nanmax(M.values))
    ax.set_xticks(range(len(vals)))
    ax.set_xticklabels([f'{v:g}' if modifier == 'age' else f'{v/1000:g}k' for v in vals], fontsize=8.2)
    ax.set_yticks(range(len(labels))); ax.set_yticklabels([l.replace('>=', '≥') for l in labels], fontsize=8.2)
    ax.set_xlabel('Vessel age (years)' if modifier == 'age' else 'Gross tonnage', fontsize=8.8, color=INK)
    ax.tick_params(length=0, colors=GREY)
    for s in ax.spines.values(): s.set_visible(False)
    thr = 0.6 * np.nanmax(M.values)
    for i in range(len(labels)):
        for j in range(len(vals)):
            v = M.values[i, j]; tc = 'white' if v > thr else INK
            if annot:
                ax.text(j, i-.13, f'{v:.1f}', ha='center', va='center', fontsize=8, fontweight='bold', color=tc)
                ax.text(j, i+.22, f'({LO.values[i,j]:.1f}–{HI.values[i,j]:.1f})', ha='center', va='center', fontsize=5.9, color=tc)
            else:
                ax.text(j, i, f'{v:.1f}', ha='center', va='center', fontsize=8, color=tc)
    cb = fig.colorbar(im, ax=ax, fraction=.04, pad=.02); cb.outline.set_visible(False)
    cb.ax.tick_params(labelsize=7, colors=GREY)
    if annot:
        cb.set_label(f'{OUTCOMES[y]} (%)', fontsize=7.8, color=INK)
        src = T_AGE if modifier == 'age' else T_SIZE
        pj = src[(src.hazard == hz) & (src.outcome == y)]['joint_interaction_p'].iloc[0]
        ax.set_title(f'{OUTCOMES[y]}     $P_{{interaction}}$ {p_text(pj)}', fontsize=9.2, loc='left', pad=6, color=INK)

def show_pred_row(hz, modifier, panel_labels, annot=True):
    table = T_PRED_AGE if modifier == 'age' else T_PRED_SIZE
    fig, axes = plt.subplots(1, 2, figsize=(13.4, 3.9), dpi=150)
    for ax, y, pl in zip(axes, ['tloss', 'ser'], panel_labels):
        _heat(ax, fig, table, hz, y, modifier, annot)
        if annot:
            ax.text(-.16, 1.10, pl, transform=ax.transAxes, fontsize=13, fontweight='bold', color=INK)
    axes[0].set_ylabel(HAZARDS[hz]['symbol'], fontsize=9, color=INK)
    if annot:
        fig.suptitle(f'{HAZARDS[hz]["name"]} × {"vessel age" if modifier == "age" else "vessel size"}',
                     x=.01, ha='left', fontsize=10.5, color=INK)
    fig.tight_layout(); plt.show()

for annot in (True, False):
    show_pred_row('wv', 'age',  ('a', 'b'), annot)     # waves x vessel age: total loss | serious
    show_pred_row('wd', 'age',  ('c', 'd'), annot)     # wind x vessel age
    show_pred_row('wv', 'size', ('e', 'f'), annot)     # waves x tonnage
    show_pred_row('wd', 'size', ('g', 'h'), annot)     # wind x tonnage


In [ ]:
# Typical vessel profiles: vessel age at p10/p90, GT at p10/p90
AGE_LO, AGE_HI = [float(round(v)) for v in g['age'].quantile([.10, .90])]
GT_LO,  GT_HI  = [float(round(v, -2)) for v in g['GT'].quantile([.10, .90])]
PROFILES = [  # (label, age, GT, colour role, linestyle)
    ('Young, large', AGE_LO, GT_HI, 'light', '-'),
    ('Young, small', AGE_LO, GT_LO, 'light', '--'),
    ('Old, large',   AGE_HI, GT_HI, 'dark',  '-'),
    ('Old, small',   AGE_HI, GT_LO, 'dark',  '--'),
]
print(f'Profiles: vessel age {AGE_LO:g} / {AGE_HI:g} yr, GT {GT_LO:,.0f} / {GT_HI:,.0f}')

def make_cf_profile(hz, lab, age, gt):
    cf = g.copy()
    cf[hz] = pd.Categorical([lab]*len(cf), categories=HAZARDS[hz]['labels'], ordered=True)
    cf['age5c'] = (age - age_ref) / 5.0
    cf['lgtc']  = np.log(gt / gt_ref)
    return cf

rows = []
for hz, meta in HAZARDS.items():
    for y in OUTCOMES:
        res = full_models[(y, hz)]
        for plab, a, gt, _, _ in PROFILES:
            for i, lab in enumerate(meta['labels']):
                rows.append(dict(hazard=hz, outcome=y, profile=plab, age=a, GT=gt, category=lab,
                                 category_index=i, **avg_prediction(res, make_cf_profile(hz, lab, a, gt))))
T_PROFILE = pd.DataFrame(rows)
T_PROFILE.to_csv(OUT_DIR / 'table_predicted_by_vessel_profile.csv', index=False)
print(T_PROFILE.pivot_table(index=['hazard','outcome','category'], columns='profile', values='b').round(1).to_string())

def _profile_panel(ax, hz, y, annot):
    meta = HAZARDS[hz]; labels = meta['labels']; x = np.arange(len(labels))
    for i in range(0, len(labels), 2): ax.axvspan(i-.5, i+.5, color=LIGHT_GREY, lw=0, zorder=0)
    dodge = np.linspace(-.18, .18, len(PROFILES))
    for k, (plab, a, gt, role, ls) in enumerate(PROFILES):
        col = meta['colour_light'] if role == 'light' else meta['colour_dark']
        d = T_PROFILE[(T_PROFILE.hazard == hz) & (T_PROFILE.outcome == y)
                      & (T_PROFILE.profile == plab)].sort_values('category_index')
        xx = x + dodge[k]
        ax.plot(xx, d['b'], ls=ls, color=col, lw=1.6, zorder=2)
        ax.errorbar(xx, d['b'], yerr=[d['b']-d['lo'], d['hi']-d['b']], fmt='o', ms=4, color=col,
                    mfc=col if ls == '-' else 'white', mew=1.2, elinewidth=.9, capsize=0, zorder=3)
        if annot:
            ax.text(xx[-1] + .12, d['b'].iloc[-1], f'{d["b"].iloc[-1]:.0f}', va='center', ha='left',
                    fontsize=6.8, color=col)
    ax.set_xticks(x); ax.set_xticklabels([l.replace('>=', '≥') for l in labels], fontsize=8.2)
    ax.set_xlim(-.6, len(labels) - .2)
    ax.set_xlabel(meta['symbol'], fontsize=9, color=INK)
    ax.set_ylabel(f'Predicted probability of\n{OUTCOMES[y].lower()} (%)', fontsize=8.8, color=INK)
    ax.grid(axis='y', color='#e5e8ea', lw=.5)
    if annot:
        t = T_FULL_TESTS[(T_FULL_TESTS.hazard == hz) & (T_FULL_TESTS.outcome == y)].set_index('modifier')
        ax.text(.02, .97, f'$P_{{age}}$ {p_text(t.loc["age5c","joint_interaction_p"])}   '
                          f'$P_{{size}}$ {p_text(t.loc["lgtc","joint_interaction_p"])}',
                transform=ax.transAxes, ha='left', va='top', fontsize=7.4, color=INK)
    style_axis(ax)

def show_profile_figure(annot=True):
    fig, axes = plt.subplots(2, 2, figsize=(11.6, 8.0), dpi=150)
    for r, hz in enumerate(['wv', 'wd']):
        for c, y in enumerate(['tloss', 'ser']):
            ax = axes[r, c]; _profile_panel(ax, hz, y, annot)
            if annot:
                ax.text(-.16, 1.06, 'abcd'[r*2+c], transform=ax.transAxes, fontsize=13, fontweight='bold', color=INK)
                if c == 0: ax.set_title(HAZARDS[hz]['name'], fontsize=10, loc='left', pad=8, color=INK)
    if annot:
        h = [Line2D([], [], color='#555', ls='-',  lw=1.6, marker='o', ms=4, mfc='#555', label=f'Large ({GT_HI/1000:g}k GT)'),
             Line2D([], [], color='#555', ls='--', lw=1.6, marker='o', ms=4, mfc='white', label=f'Small ({GT_LO/1000:g}k GT)'),
             Line2D([], [], color='#b8b8b8', lw=6, label=f'Young ({AGE_LO:g} yr)'),
             Line2D([], [], color='#4a4a4a', lw=6, label=f'Old ({AGE_HI:g} yr)')]
        fig.legend(handles=h, loc='lower center', bbox_to_anchor=(.5, .01), ncol=4, frameon=False, fontsize=8.4)
    fig.tight_layout(rect=(0, .05 if annot else 0, 1, 1)); plt.show()

show_profile_figure(annot=True)
show_profile_figure(annot=False)
